# 🥉 Camada Bronze: Ingestão e Leitura

In [0]:
# Importando as funções extras do PySpark que vamos usar na limpeza
from pyspark.sql.functions import col, split

# Extraindo os dados da nossa Tabela Bronze
df_bronze = spark.table("default.tabela_clubes")

# Mostrando as primeiras 5 linhas para ver como o dado está
display(df_bronze.limit(5))

Ano,Pos.,Clubes,Vitorias,Derrotas,Empates,GolsF/S,Saldo,Qtd_Jogadores,Idade_Media,Estrangeiros,Valor_total,Media_Valor
2017,18,America-MG,10,10,18,30:47,-17,51,"24,8",0,27350000,536000
2017,7,Athletico-PR,16,9,13,54:37,17,52,24,3,37650000,724000
2017,6,Atletico-MG,17,8,13,56:43,13,50,"23,4",6,61350000,1230000
2017,11,Bahia,12,12,14,39:41,-2,48,"23,2",2,34900000,727000
2017,9,Botafogo,13,12,13,38:46,-8,45,"23,1",4,25550000,568000


# 🥈 Camada Silver: Limpeza e Transformação

In [0]:
# Importando os tipos numéricos necessários para a conversão
from pyspark.sql.types import IntegerType, DoubleType

# INICIANDO A TRANSFORMAÇÃO (Criando o DataFrame da Camada Silver)
# Pegamos o dataframe de bronze e começamos a encadear as mudanças:

df_silver = df_bronze \
    .withColumnRenamed("Pos.", "Posicao") \
    .withColumn("Ano", col("Ano") + 1) \
    .withColumn("GolsFeitos", split(col("GolsF/S"), ":")[0].cast(IntegerType())) \
    .withColumn("GolsSofridos", split(col("GolsF/S"), ":")[1].cast(IntegerType())) \
    .withColumn("Valor_total", col("Valor_total").cast(DoubleType())) \
    .withColumn("Media_Valor", col("Media_Valor").cast(DoubleType())) \
    .drop("GolsF/S") # Apagando a coluna original bagunçada, já que a separamos

# Mostrando o resultado da limpeza
display(df_silver.limit(5))

Ano,Posicao,Clubes,Vitorias,Derrotas,Empates,Saldo,Qtd_Jogadores,Idade_Media,Estrangeiros,Valor_total,Media_Valor,GolsFeitos,GolsSofridos
2018,18,America-MG,10,10,18,-17,51,"24,8",0,2.735E7,536000.0,30,47
2018,7,Athletico-PR,16,9,13,17,52,24,3,3.765E7,724000.0,54,37
2018,6,Atletico-MG,17,8,13,13,50,"23,4",6,6.135E7,1230000.0,56,43
2018,11,Bahia,12,12,14,-2,48,"23,2",2,3.49E7,727000.0,39,41
2018,9,Botafogo,13,12,13,-8,45,"23,1",4,2.555E7,568000.0,38,46


In [0]:
# FINALIZANDO O ETL: LOAD (Carga)
# Salvando o nosso DataFrame limpo como uma tabela permanente no formato Delta

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.silver_tabela_clubes")

print("Camada Silver criada e salva com sucesso!")

Camada Silver criada e salva com sucesso!


# 🥇 Camada Gold: Análises e Respostas de Negócio

In [0]:
%sql
-- CAMADA GOLD: Análise 1
-- Pergunta: Qual a média de valor de elenco para cada posição na tabela?

SELECT 
    Posicao, 
    ROUND(AVG(Valor_total), 2) AS Media_Valor_Total_Elenco
FROM default.silver_tabela_clubes
GROUP BY Posicao
ORDER BY Posicao ASC

-- A resposta é SIM, o dinheiro tem um impacto gigantesco! Repare que há uma clara tendência de queda da esquerda para a direita. Os times que ocupam as posições de 1º a 4º lugar possuem elencos com médias de valor absurdamente mais altas (na casa dos 50 a 60 milhões). Já os times na zona de rebaixamento (17º ao 20º) amargam os elencos mais baratos (na casa dos 20 milhões).

Posicao,Media_Valor_Total_Elenco
1,5.2493E7
2,5.2044E7
3,5.4485E7
4,6.287E7
5,4.6122E7
6,5.0411E7
7,3.2791E7
8,4.4481E7
9,5.0275E7
10,5.0543E7


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- CAMADA GOLD: Análise 2
-- Pergunta: Experiência vs. Juventude: Qual o saldo de gols médio agrupado por idade do elenco?

SELECT 
    -- 1. Troca a vírgula por ponto, transforma em número e arredonda (ex: 24,8 vira 25)
    ROUND(CAST(REPLACE(Idade_Media, ',', '.') AS FLOAT), 0) AS Idade_Aproximada,
    
    -- 2. Calcula a média do saldo de gols para aquela idade
    ROUND(AVG(Saldo), 2) AS Media_Saldo_Gols,
    
    -- 3. Conta quantos times tinham essa idade para termos contexto
    COUNT(*) AS Quantidade_de_Times
FROM default.silver_tabela_clubes
GROUP BY Idade_Aproximada
ORDER BY Idade_Aproximada ASC

-- Os dados mostram um resultado surpreendente! A juventude leva a melhor. Times com média de idade entre 22 e 23 anos possuem um saldo de gols médio positivo (fazem mais gols do que sofrem). Conforme o elenco envelhece (25 anos para cima), o saldo de gols médio despenca e fica negativo. Isso sugere que times mais jovens têm mais fôlego e velocidade para manter um saldo positivo ao longo de um campeonato de pontos corridos.

Idade_Aproximada,Media_Saldo_Gols,Quantidade_de_Times
22.0,2.33,6
23.0,2.18,56
24.0,0.3,77
25.0,-1.82,44
26.0,-4.31,16
27.0,-10.0,1


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- CAMADA GOLD: Análise 3
-- Pergunta: O Fator Gringo: Ter mais estrangeiros no elenco significa mais vitórias?

SELECT 
    Estrangeiros AS Quantidade_Estrangeiros,
    ROUND(AVG(Vitorias), 1) AS Media_Vitorias,
    COUNT(*) AS Total_Times
FROM default.silver_tabela_clubes
GROUP BY Estrangeiros
ORDER BY Estrangeiros ASC

-- Os dados revelam que o 'Fator Gringo' tem, sim, um impacto positivo no desempenho das equipes. Podemos observar uma tendência de alta na média de vitórias conforme a quantidade de estrangeiros aumenta de 0 para 4. Curiosamente, a média de vitórias atinge o seu pico (16.2 vitórias) nos times que possuem exatamente 4 estrangeiros no elenco. A partir de 5 estrangeiros, a média sofre uma leve queda, sugerindo que existe um "ponto de equilíbrio" ideal para mesclar talentos internacionais com a base nacional.

Quantidade_Estrangeiros,Media_Vitorias,Total_Times
0,11.6,30
1,12.6,32
2,14.0,27
3,15.0,32
4,16.2,34
5,12.5,22
6,13.7,13
7,15.8,9
8,15.0,1


Databricks visualization. Run in Databricks to view.